# Raw MES Data Validation

This notebook validates the raw historical MES trade dataset before any feature engineering or modeling.

The goals are to confirm:

- the raw DBN file can be opened successfully
- the dataset uses the expected `trades` schema
- the total number of trade records
- the exact historical date coverage
- the underlying MES contracts represented by the continuous `MES.v.0` series
- the contract roll dates
- Databento data-quality warnings such as degraded dates
- basic sanity checks for timestamps, prices, and trade sizes

The raw `.dbn` file is treated as immutable source data. Any cleaned, aggregated, or model-ready data will be created separately.

In [2]:
# Import Databento and open the full raw MES Trades file.
# This opens the DBN file so we can inspect it without loading all
# 99 million individual trades into memory at once.

import databento as db

raw_file_path = "../data/mes_trades_2025-10-07_to_2026-09-11.dbn"

full_data = db.DBNStore.from_file(raw_file_path)

print(full_data)

<DBNStore(schema=trades)>


## Dataset Coverage and Record Count

This section verifies the total number of trade records and the exact time period covered by the raw dataset.

Because the dataset contains nearly 100 million individual trades, the file is processed sequentially rather than converted into one large pandas DataFrame. This avoids loading the entire dataset into memory at once.

In [3]:
# Count every trade record and capture the first and last event timestamps.
# We iterate through the DBN file rather than loading all trades into memory.
# This may take some time because the file contains roughly 99 million records.

record_count = 0
first_timestamp = None
last_timestamp = None

for record in full_data:
    if first_timestamp is None:
        first_timestamp = record.ts_event

    last_timestamp = record.ts_event
    record_count += 1

print(f"Total trade records: {record_count:,}")
print("First event timestamp:", first_timestamp)
print("Last event timestamp:", last_timestamp)

Total trade records: 99,319,450
First event timestamp: 1759795200043782871
Last event timestamp: 1789084799790465427


### Human-Readable Date Coverage

Databento stores event timestamps as nanoseconds since the Unix epoch. The first and last timestamps are converted to UTC datetimes below so the exact dataset coverage can be verified.

In [4]:
# Convert only the first and last event timestamps into human-readable UTC datetimes.
# This does not convert or modify the 99 million trades in the raw DBN file.

import pandas as pd

first_datetime = pd.to_datetime(first_timestamp, unit="ns", utc=True)
last_datetime = pd.to_datetime(last_timestamp, unit="ns", utc=True)

print("First trade:", first_datetime)
print("Last trade:", last_datetime)

First trade: 2025-10-07 00:00:00.043782871+00:00
Last trade: 2026-09-10 23:59:59.790465427+00:00


## Continuous Contract Mapping

The historical data was downloaded using Databento's `MES.v.0` continuous contract symbol.

`MES.v.0` follows the highest-volume MES futures contract over time rather than representing one individual futures contract. The mapping below identifies when the continuous series rolled from one underlying quarterly MES contract to the next.

In [5]:
# Inspect the continuous-contract mapping stored in the DBN metadata.
# This reads the small symbology mapping and does not scan all 99 million trades.

print(full_data.symbology)

{'symbols': ['MES.v.0'], 'stype_in': 'continuous', 'stype_out': 'instrument_id', 'start_date': '2025-10-07', 'end_date': '2026-09-11', 'partial': [], 'not_found': [], 'mappings': {'MES.v.0': [{'start_date': datetime.date(2025, 10, 7), 'end_date': datetime.date(2025, 12, 17), 'symbol': '42004164'}, {'start_date': datetime.date(2025, 12, 17), 'end_date': datetime.date(2026, 3, 18), 'symbol': '42003800'}, {'start_date': datetime.date(2026, 3, 18), 'end_date': datetime.date(2026, 6, 17), 'symbol': '42005163'}, {'start_date': datetime.date(2026, 6, 17), 'end_date': datetime.date(2026, 9, 11), 'symbol': '42003239'}]}}


In [7]:
# Load the Databento API key from the private .env file and create
# an authenticated Historical API client for metadata/symbology requests.
# The API key itself is never printed or stored in the notebook.

import os
from dotenv import load_dotenv

load_dotenv("../.env")

client = db.Historical(key=os.getenv("DATABENTO_API_KEY"))

print("Databento client ready.")

Databento client ready.


In [8]:
# Resolve the Databento instrument IDs used by MES.v.0 into actual CME contract symbols.
# This uses Databento's symbology service and does not scan the full trade dataset.

contract_ids = [
    "42004164",
    "42003800",
    "42005163",
    "42003239",
]

contract_symbols = client.symbology.resolve(
    dataset="GLBX.MDP3",
    symbols=contract_ids,
    stype_in="instrument_id",
    stype_out="raw_symbol",
    start_date="2025-10-07",
    end_date="2026-09-11",
)

print(contract_symbols)

{'result': {'42004164': [{'d0': '2025-10-07', 'd1': '2026-01-04', 's': 'MESZ5'}, {'d0': '2026-01-04', 'd1': '2026-06-01', 's': 'ZYECK6'}, {'d0': '2026-06-01', 'd1': '2026-08-16', 's': '1SN608'}, {'d0': '2026-08-16', 'd1': '2026-09-11', 's': 'NDKX0'}], '42003800': [{'d0': '2025-10-07', 'd1': '2026-09-11', 's': 'MESH6'}], '42005163': [{'d0': '2025-10-07', 'd1': '2026-07-02', 's': 'MESM6'}, {'d0': '2026-07-02', 'd1': '2026-08-30', 's': 'NWDU609'}, {'d0': '2026-08-30', 'd1': '2026-09-11', 's': 'ESXZ6'}], '42003239': [{'d0': '2025-10-07', 'd1': '2026-09-11', 's': 'MESU6'}]}, 'symbols': ['42004164', '42003800', '42005163', '42003239'], 'stype_in': 'instrument_id', 'stype_out': 'raw_symbol', 'start_date': '2025-10-07', 'end_date': '2026-09-11', 'partial': [], 'not_found': [], 'message': 'OK', 'status': 0}


### Verified Contract Sequence

The continuous `MES.v.0` series mapped to the following quarterly MES futures contracts during the historical sample:

| Period | Instrument ID | MES Contract |
|---|---:|---|
| 2025-10-07 → 2025-12-17 | 42004164 | MESZ5 |
| 2025-12-17 → 2026-03-18 | 42003800 | MESH6 |
| 2026-03-18 → 2026-06-17 | 42005163 | MESM6 |
| 2026-06-17 → 2026-09-11 | 42003239 | MESU6 |

This confirms that the continuous series rolled through the expected December 2025, March 2026, June 2026, and September 2026 MES contracts.

## Data Quality

Databento provides a daily condition status for its historical datasets. This section checks the full historical period for dates that Databento identified as degraded, missing, or otherwise not fully available.

A degraded date does not necessarily mean that all MES trades on that date are unusable. These dates are identified so they can be investigated and handled appropriately during data processing.

In [9]:
# Check Databento's official data-quality status for every calendar date
# covered by our historical MES dataset.
# Only dates that are not marked "available" are displayed.

dataset_conditions = client.metadata.get_dataset_condition(
    dataset="GLBX.MDP3",
    start_date="2025-10-07",
    end_date="2026-09-10",
)

problem_dates = [
    day for day in dataset_conditions
    if day["condition"] != "available"
]

print(f"Total dates checked: {len(dataset_conditions)}")
print(f"Dates not fully available: {len(problem_dates)}")

for day in problem_dates:
    print(day["date"], "-", day["condition"])

Total dates checked: 331
Dates not fully available: 9
2025-11-28 - degraded
2026-01-31 - degraded
2026-03-15 - degraded
2026-03-16 - degraded
2026-03-21 - degraded
2026-04-10 - degraded
2026-05-24 - degraded
2026-07-30 - degraded
2026-08-29 - degraded


## Validation Summary

The raw MES historical dataset passed the initial validation checks:

- **Schema:** Databento `trades`
- **Total trade records:** 99,319,450
- **First trade:** 2025-10-07 00:00:00.043782871 UTC
- **Last trade:** 2026-09-10 23:59:59.790465427 UTC
- **Continuous contract:** `MES.v.0`
- **Underlying contracts:** MESZ5 → MESH6 → MESM6 → MESU6
- **Databento dates checked:** 331
- **Degraded dates:** 9
- **Missing dates identified:** 0

### Data-Processing Decisions

The original DBN file will remain unchanged and serve as the immutable source dataset.

The nine degraded dates will be retained for now rather than automatically removed. They will be flagged during processing so their effect on the MES data can be investigated before deciding whether any affected observations or sessions should be excluded.

Contract-roll boundaries will also be identified during processing so that price changes caused by switching underlying futures contracts are not mistakenly interpreted as normal market returns.

Because the raw dataset contains approximately 99 million trades, subsequent processing will be performed incrementally rather than loading the entire dataset into memory at once.